Python notebook to execute MACE algorithm on (CliRe)NER data. Please acquire [mace.py](https://github.com/dirkhovy/MACE/blob/master/mace.py).

In [3]:
import os
import csv
import json
import subprocess
import tempfile
import re
from collections import defaultdict
import uuid

# --- IMPORTS FROM YOUR MODULES ---
try:
    from dataset_processing import (
        cwed4eta_process_json_file,
        convert_to_token_spans,
        process_directory_of_json_files
    )
except ImportError:
    print("⚠️ Warning: Could not import dataset_processing.")

# ==========================================
# 1. CONFIGURATION
# ==========================================
ANNOTATOR_DIR = "/home/p0l3/RAD/DROP/CLIRENER/ANNOTATORS/"
MACE_SCRIPT_PATH = "mace.py"  # Ensure this file is in the same folder
OUTPUT_JSON = "mace_consensus_output.json"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def spans_to_bio_tags(token_list, ner_spans):
    """Converts spans to BIO tags. Handles both Dict and List formats."""
    tags = ["O"] * len(token_list)
    if not ner_spans:
        return tags
    
    for item in ner_spans:
        # Robust unpacking
        if isinstance(item, dict):
            start = item['start']
            end = item['end']
            label = item['label']
        else:
            # Assume List [start, end_inclusive, label, ...]
            start = item[0]
            end = item[1]
            label = item[2]
            
        b_tag = f"B-{label}"
        i_tag = f"I-{label}"
        
        tags[start] = b_tag
        if end > start:
            # Range is exclusive at the top, so +1
            for i in range(start + 1, end + 1): 
                tags[i] = i_tag
    return tags

def bio_tags_to_spans(tags):
    """
    Standard conversion for MACE output.
    Returns: [start, end_inclusive, label]
    """
    spans = []
    start = -1
    label = ""
    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start != -1:
                spans.append([start, i - 1, label])
            start = i
            label = tag[2:]
        elif tag.startswith("I-"):
            if start == -1: start = i; label = tag[2:]
            elif tag[2:] != label:
                spans.append([start, i - 1, label])
                start = i
                label = tag[2:]
        else:
            if start != -1:
                spans.append([start, i - 1, label])
            start = -1; label = ""
    if start != -1:
        spans.append([start, len(tags) - 1, label])
    return spans

def load_data_raw():
    print(f"📂 Loading data from: `{ANNOTATOR_DIR}`")
    
    DATA_CONFIG = {
        0: None, # Explicitly requested as None
        1:  {"path": "1/G3_10226.json",   "ids": [4, 1, 5]},
        2:  {"path": "2/G4_5326.json",   "ids": [1, 6]},
        3:  {"path": "3/G4_4326.json",   "ids": [1, 7]},
        4:  {"path": "4/G2_5326_1.json",    "ids": [1, 8]},
        5:  {"path": "5/",                "ids": [1, 9], "is_dir": True},
        6:  {"path": "6/G1_15126.json",   "ids": [13]},
        7:  None,
        8:  {"path": "8/G6_5326.json",    "ids": [12]},
        9:  {"path": "9/G2_5326.json",    "ids": [1, 11]},
        10: {"path": "10/G3_10226_2.json",  "ids": [1, 15]},
        11: {"path": "11/G5_21126.json",  "ids": [4, 1, 14]},
        12: {"path": "12/G5_4326.json",  "ids": [4, 1, 2, 3]},
    }
    def safe_load(subpath, annotator_ids, is_directory=False):
        full_path = os.path.join(ANNOTATOR_DIR, subpath)
        if not os.path.exists(full_path):
            print(f"❌ Path not found: {full_path}")
            return None
        try:
            if is_directory:
                raw_data = process_directory_of_json_files(full_path, annotator_ids)
            else:
                raw_data = cwed4eta_process_json_file(full_path, annotator_ids)
            return convert_to_token_spans(raw_data)
        except Exception as e:
            print(f"⚠️ Error loading {subpath}: {e}")
            return None

    all_data = []
    for i in range(13): # To fix annotator 12 that is directly loaded from a processed json
        config = DATA_CONFIG.get(i)
        
        if config is None:
            all_data.append(None)
        else:
            if i == 12:
                with open(ANNOTATOR_DIR + config["path"], "r") as f:
                    loaded_data = json.load(f)
            else:
                loaded_data = safe_load(
                    config["path"], 
                    config["ids"], 
                    is_directory=config.get("is_dir", False)
                )
            all_data.append(loaded_data)
            
    return all_data


def get_token_char_offsets(text, tokens):
    """
    Reconstructs character start/end for tokens based on the raw text.
    Returns a list of tuples: [(start_char, end_char), ...]
    """
    offsets = []
    current_pos = 0
    text_lower = text.lower()
    
    for token in tokens:
        # Simple finder - assumes tokens appear in order
        # You might need a more robust tokenizer alignment if your tokens are drastically normalized
        token_str = token.lower()
        start = text_lower.find(token_str, current_pos)
        
        if start == -1:
            # Fallback: if token not found (e.g. specialized chars), skip or estimate
            # This is a common pain point. We'll assume strict alignment for now.
            offsets.append((current_pos, current_pos + len(token)))
            current_pos += len(token)
        else:
            end = start + len(token)
            offsets.append((start, end))
            current_pos = end
            
    return offsets


def convert_consensus_to_labelstudio(consensus_data, model_version="Consensus_Voting_v1"):
    """
    Args:
        consensus_data: List of dicts from generate_consensus_dataset (v1 or v2)
    """
    labelstudio_data = []
    
    for doc in consensus_data:
        text = doc['text']
        tokens = doc['tokenized_text']
        alpha_score = doc.get('alpha')
        num_raters = doc.get('num_raters', 0)
        raw_id = str(doc['id'])
        
        # Parse Paper ID and Sentence ID
        if '-' in raw_id:
            paper_id, sentence_id = raw_id.rsplit('-', 1)
        else:
            paper_id = raw_id
            sentence_id = "N/A"

        # Generate mapping from token index to character offsets
        # Ensure get_token_char_offsets is available in your scope
        token_char_map = get_token_char_offsets(text, tokens)
        
        results = []
        
        # Iterate through spans using dynamic unpacking to handle V1 (List) and V2 (Dict/List+Stats)
        for item in doc['ner']:
            vote_stats = None
            
            # --- CASE 1: Item is a Dictionary (Likely V2 from Visualizer) ---
            if isinstance(item, dict):
                start_tok = item['start']
                end_tok = item['end']
                label = item['label']
                is_tie = item.get('tie', False)
                vote_stats = item.get('vote_stats', None)
            
            # --- CASE 2: Item is a List/Tuple (Likely V1 or V2 backend) ---
            else:
                # Basic unpacking
                start_tok = item[0]
                end_tok = item[1]
                label = item[2]
                is_tie = item[3] if len(item) > 3 else False
                
                # Check for V2 extra data (vote_stats) in the list
                if len(item) > 4:
                    vote_stats = item[4]

            # --- Calculate Character Offsets ---
            # Safety check to ensure indices are within bounds of the char map
            if start_tok < len(token_char_map) and end_tok < len(token_char_map):
                start_char = token_char_map[start_tok][0]
                end_char = token_char_map[end_tok][1]
                chunk_text = text[start_char:end_char]
                
                # Build Meta Data (Include ties and detailed stats if available)
                meta_data = {
                    "is_tie": is_tie,
                    "score": 0.5 if is_tie else 1.0 
                }
                if vote_stats:
                    meta_data["vote_stats"] = vote_stats

                results.append({
                    "id": str(uuid.uuid4())[:8],
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": start_char,
                        "end": end_char,
                        "text": chunk_text,
                        "labels": [label]
                    },
                    "meta": meta_data
                })
            
        prediction_entry = {
            "model_version": model_version,
            "score": 1.0, 
            "result": results
        }

        # Format Alpha for display
        if alpha_score is None or np.isnan(alpha_score):
            display_alpha = "N/A"
        else:
            display_alpha = float(round(alpha_score, 4))

        ls_entry = {
            "data": {
                "sentence": text,
                "id": raw_id,
                "paper_id": paper_id,        
                "sentence_id": sentence_id,  
                "num_raters": num_raters,    
                "alpha_agreement": display_alpha
            },
            "predictions": [prediction_entry]
        }
        
        labelstudio_data.append(ls_entry)
        
    return labelstudio_data



# ==========================================
# 3. MACE PIPELINE
# ==========================================

def run_mace_pipeline(annotator_data_list):
    if not os.path.exists(MACE_SCRIPT_PATH):
        raise FileNotFoundError(f"Could not find {MACE_SCRIPT_PATH}")

    # 1. Identify Active Annotators
    active_indices = [i for i, d in enumerate(annotator_data_list) if d is not None]
    
    print(f"\n[1/4] Preparing Data...")
    print(f"      Active Annotator Indices: {active_indices}")

    # 2. Align Documents
    aligned_docs = defaultdict(dict)
    ref_docs = {}

    for original_idx in active_indices:
        dataset = annotator_data_list[original_idx]
        for doc in dataset:
            doc_id = doc['id']
            aligned_docs[doc_id][original_idx] = doc
            if doc_id not in ref_docs:
                ref_docs[doc_id] = {
                    'text': doc['text'],
                    'tokenized_text': doc['tokenized_text']
                }
    
    all_doc_ids = sorted(aligned_docs.keys())
    
    # --- FILTERING STEP ---
    # Only keep docs with >= 2 annotators
    valid_doc_ids = []
    skipped_count = 0
    
    for doc_id in all_doc_ids:
        # Check how many annotators are present for this specific document ID
        annotator_count = len(aligned_docs[doc_id])
        
        if annotator_count >= 2:
            valid_doc_ids.append(doc_id)
        else:
            skipped_count += 1
            
    print(f"      Total Documents Found: {len(all_doc_ids)}")
    print(f"      Skipped (single annotator): {skipped_count}")
    print(f"      Processing {len(valid_doc_ids)} valid documents.")

    if not valid_doc_ids:
        print("❌ No documents have >= 2 annotators. Exiting.")
        return []

    # 3. Build CSV Rows
    mace_rows = []
    token_map = [] # Track which doc each token row belongs to

    for doc_id in valid_doc_ids:
        tokens = ref_docs[doc_id]['tokenized_text']
        doc_len = len(tokens)
        token_map.append({'doc_id': doc_id, 'length': doc_len})
        
        # Get tags for ACTIVE annotators only
        row_block = []
        for t_i in range(doc_len):
            row_block.append([]) # Init row

        for active_idx in active_indices:
            # Check if this specific annotator saw this doc
            if active_idx in aligned_docs[doc_id]:
                # Get their tags
                doc_data = aligned_docs[doc_id][active_idx]
                tags = spans_to_bio_tags(tokens, doc_data['ner'])
            else:
                # They exist in the project, but skipped this doc -> Empty String
                tags = [""] * doc_len
            
            # Append to rows
            for t_i in range(doc_len):
                row_block[t_i].append(tags[t_i])
        
        mace_rows.extend(row_block)

    print(f"      Total tokens: {len(mace_rows)}")
    
    # 4. Execute MACE
    print("\n[2/4] Running MACE...")
    tmpdirname = "/home/p0l3/RAD/CLIRENER/CliReNER/MACE_RESULTS/"
    #with tempfile.TemporaryDirectory() as tmpdirname:
    csv_path = os.path.join(tmpdirname, "input.csv")
    prefix_path = os.path.join(tmpdirname, "mace_out")
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerows(mace_rows)
        
    # Run MACE
    # Using 50 iterations should be enough for convergence
    cmd = ["python3", MACE_SCRIPT_PATH, "--prefix", prefix_path, "--iterations", "50", csv_path]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("❌ MACE crashed!")
        print(result.stderr)
        return []
        
    print("      MACE Finished.")

    # 5. Parse Competence
    print("\n[3/4] Competence Scores (0.0 - 1.0):")
    comp_file = prefix_path + ".competence"
    if os.path.exists(comp_file):
        with open(comp_file, 'r') as f:
            comps = f.read().strip().split('\t')
            for mace_col_idx, score in enumerate(comps):
                if mace_col_idx < len(active_indices):
                    original_id = active_indices[mace_col_idx]
                    print(f"      Original Annotator {original_id}: {float(score):.4f}")

    # 6. Parse Predictions
    pred_file = prefix_path + ".prediction"
    if not os.path.exists(pred_file):
        print("❌ Prediction file not found.")
        return []
        
    with open(pred_file, 'r') as f:
        predicted_labels = [line.strip() for line in f if line.strip()]

    # 7. Reconstruct
    print("\n[4/4] Building Result JSON...")
    
    consensus_data = []
    current_idx = 0
    
    if len(predicted_labels) != len(mace_rows):
        print(f"⚠️ Warning: Output length mismatch ({len(predicted_labels)} vs {len(mace_rows)}). Truncating/Padding.")
    
    for doc_info in token_map:
        doc_id = doc_info['doc_id']
        doc_len = doc_info['length']
        
        # Safely slice
        end_idx = min(current_idx + doc_len, len(predicted_labels))
        doc_tags = predicted_labels[current_idx : end_idx]
        current_idx += doc_len
        
        # Padding if truncated
        if len(doc_tags) < doc_len:
            doc_tags += ["O"] * (doc_len - len(doc_tags))

        spans = bio_tags_to_spans(doc_tags)
        num_raters = len(aligned_docs[doc_id])
        
        consensus_data.append({
            "id": doc_id,
            "text": ref_docs[doc_id]['text'],
            "sentence": ref_docs[doc_id]['text'],
            "tokenized_text": ref_docs[doc_id]['tokenized_text'],
            "ner": spans,
            "num_raters": num_raters,
            "method": "MACE"
        })
        
    return consensus_data


This process takes 26 minutes for this setup:

```shell
[1/4] Preparing Data...
      Active Annotator Indices: [1, 2, 3, 4, 5, 6, 8, 10, 11]
      Total Documents Found: 184
      Skipped (single annotator): 42
      Processing 142 valid documents.
      Total tokens: 7625

[2/4] Running MACE...
      MACE Finished.

[3/4] Competence Scores (0.0 - 1.0):
      Original Annotator 1: 0.7985
      Original Annotator 2: 0.9011
      Original Annotator 3: 0.9525
      Original Annotator 4: 0.6493
      Original Annotator 5: 0.8683
      Original Annotator 6: 0.8130
      Original Annotator 8: 0.8372
      Original Annotator 10: 0.8677
      Original Annotator 11: 0.8398

[4/4] Building Result JSON...
```

In [4]:

if __name__ == "__main__":
    raw_data = load_data_raw()
    final_data = run_mace_pipeline(raw_data)

    # 2. Convert to Label Studio
    ls_json = convert_consensus_to_labelstudio(final_data)

# 3. Save

    with open("consensus_for_labelstudio_mace_11326.json", "w") as f:
        json.dump(ls_json, f, indent=2)

📂 Loading data from: `/home/p0l3/RAD/DROP/CLIRENER/ANNOTATORS/`

[1/4] Preparing Data...
      Active Annotator Indices: [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12]
      Total Documents Found: 192
      Skipped (single annotator): 0
      Processing 192 valid documents.
      Total tokens: 9715

[2/4] Running MACE...
      MACE Finished.

[3/4] Competence Scores (0.0 - 1.0):
      Original Annotator 1: 0.8550
      Original Annotator 2: 0.9109
      Original Annotator 3: 0.9137
      Original Annotator 4: 0.7755
      Original Annotator 5: 0.8643
      Original Annotator 6: 0.8251
      Original Annotator 8: 0.8556
      Original Annotator 9: 0.8280
      Original Annotator 10: 0.8510
      Original Annotator 11: 0.8604
      Original Annotator 12: 0.8602

[4/4] Building Result JSON...
